# 06 클러스터링 방법

**6종 임베딩** × **4종 클러스터링** = **24조합** 품질 비교 (실루엣 ↑, Davies-Bouldin ↓)

| 임베딩 | 클러스터링 |
|--------|-----------|
| PCA, FastDTW, AE, GAF-CNN, TS2Vec, PatchTST | KMeans, HAC, GMM, DBSCAN |

최적 조합 라벨 → `ml_cluster_type_family.parquet` 저장


In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))

from utils.paths import DATA_PROCESSED, ML_CLUSTER
from utils.clustering_experiments import run_embedding_clustering_grid, select_best_combo

dfw = pd.read_parquet(DATA_PROCESSED / 'df_weekly.parquet')
pivot = dfw.pivot_table(index=['type', 'family'], columns='yearweek', values='sales', fill_value=0)
meta = pivot.index.to_frame(index=False)
X_raw = pivot.values.astype(float)
K = 4
N_COMPONENTS = 10
print('series:', X_raw.shape[0], '| weeks:', X_raw.shape[1], '| K:', K)



series: 165 | weeks: 242 | K: 4


In [2]:
# 6 임베딩 × 4 클러스터링 = 24조합 (FastDTW·AE·GAF-CNN·TS2Vec·PatchTST 포함)
quality_df, label_cache = run_embedding_clustering_grid(
    X_raw, k=K, n_components=N_COMPONENTS,
)
quality_sorted = quality_df.sort_values(
    ['silhouette', 'davies_bouldin'], ascending=[False, True]
).reset_index(drop=True)
print('=== 전체 24조합 품질 (실루엣↑, DB Index↓) ===')
display(quality_sorted.round(4))



[embedding] PCA


  PCA+KMeans: silhouette=0.49409916553149, db=0.6896313697646943, n_clusters=4
  PCA+HAC: silhouette=0.8564530877738853, db=0.7436324163163133, n_clusters=4
  PCA+GMM: silhouette=0.2473123147734971, db=1.3003375769669379, n_clusters=4
  PCA+DBSCAN: silhouette=nan, db=nan, n_clusters=1
[embedding] FastDTW


  FastDTW+KMeans: silhouette=0.8123490350068091, db=0.5188366934371917, n_clusters=4
  FastDTW+HAC: silhouette=0.8173942337871033, db=0.5003616061201555, n_clusters=4
  FastDTW+GMM: silhouette=0.8123490350068091, db=0.5188366934371917, n_clusters=4
  FastDTW+DBSCAN: silhouette=0.7339057572674218, db=0.27805358777026506, n_clusters=2
[embedding] AE


  AE+KMeans: silhouette=0.7963919043540955, db=0.6097443857229788, n_clusters=4
  AE+HAC: silhouette=0.7925949692726135, db=0.5120502744355344, n_clusters=4
  AE+GMM: silhouette=0.7429206371307373, db=0.6733875807957819, n_clusters=4
  AE+DBSCAN: silhouette=0.7592741847038269, db=0.21987444421650618, n_clusters=2
[embedding] GAF-CNN


  GAF-CNN+KMeans: silhouette=0.3504122495651245, db=1.279425501517148, n_clusters=4
  GAF-CNN+HAC: silhouette=0.32207757234573364, db=1.3423050556149316, n_clusters=4
  GAF-CNN+GMM: silhouette=0.16966792941093445, db=2.720759321129086, n_clusters=4
  GAF-CNN+DBSCAN: silhouette=0.46953684091567993, db=0.9566455992057846, n_clusters=2
[embedding] TS2Vec


  TS2Vec+KMeans: silhouette=0.9213484525680542, db=0.23512484295550806, n_clusters=4
  TS2Vec+HAC: silhouette=0.9213484525680542, db=0.23512484295550806, n_clusters=4
  TS2Vec+GMM: silhouette=0.8526106476783752, db=0.44344835690287376, n_clusters=4
  TS2Vec+DBSCAN: silhouette=0.7079840302467346, db=0.3514082491243517, n_clusters=3
[embedding] PatchTST


  PatchTST+KMeans: silhouette=0.5288437604904175, db=0.7744452368211412, n_clusters=4
  PatchTST+HAC: silhouette=0.5115543007850647, db=0.704922368233388, n_clusters=4
  PatchTST+GMM: silhouette=0.4256323575973511, db=0.9086568404108907, n_clusters=4
  PatchTST+DBSCAN: silhouette=0.6680657267570496, db=0.35477387047327325, n_clusters=6
=== 전체 24조합 품질 (실루엣↑, DB Index↓) ===


,method,embedding,clustering,n_clusters,silhouette,davies_bouldin
0,TS2Vec+KMeans,TS2Vec,KMeans,4,0.9213,0.2351
1,TS2Vec+HAC,TS2Vec,HAC,4,0.9213,0.2351
2,PCA+HAC,PCA,HAC,4,0.8565,0.7436
3,TS2Vec+GMM,TS2Vec,GMM,4,0.8526,0.4434
4,FastDTW+HAC,FastDTW,HAC,4,0.8174,0.5004
5,FastDTW+KMeans,FastDTW,KMeans,4,0.8123,0.5188
6,FastDTW+GMM,FastDTW,GMM,4,0.8123,0.5188
7,AE+KMeans,AE,KMeans,4,0.7964,0.6097
8,AE+HAC,AE,HAC,4,0.7926,0.5121
9,AE+DBSCAN,AE,DBSCAN,2,0.7593,0.2199


In [3]:
best = select_best_combo(quality_df)
print('최적 조합:', best['method'])
print(f"n_clusters={int(best['n_clusters'])}, silhouette={float(best['silhouette']):.4f}, davies_bouldin={float(best['davies_bouldin']):.4f}")
print('embedding:', best['embedding'], '| clustering:', best['clustering'])

labels_best = label_cache[(best['embedding'], best['clustering'])]
out = meta.copy()
out['ML_CLUSTER'] = labels_best + 1
out['embedding_method'] = best['embedding']
out['clustering_method'] = best['clustering']
out.to_parquet(ML_CLUSTER, index=False)
quality_df.to_csv(DATA_PROCESSED / 'clustering_quality.csv', index=False)
quality_sorted.to_csv(DATA_PROCESSED / 'clustering_quality_sorted.csv', index=False)
print('저장:', ML_CLUSTER)
out.head()



최적 조합: TS2Vec+HAC
n_clusters=4, silhouette=0.9213, davies_bouldin=0.2351
embedding: TS2Vec | clustering: HAC
저장: C:\Users\kjh\ai-retail-demandforecasting\data\processed\ml_cluster_type_family.parquet


,type,family,ML_CLUSTER,embedding_method,clustering_method
0,A,AUTOMOTIVE,1,TS2Vec,HAC
1,A,BABY CARE,1,TS2Vec,HAC
2,A,BEAUTY,1,TS2Vec,HAC
3,A,BEVERAGES,2,TS2Vec,HAC
4,A,BOOKS,1,TS2Vec,HAC


## 분석 요약

### 실험 설계
- **6 임베딩** (PCA, FastDTW, AE, GAF-CNN, TS2Vec, PatchTST) × **4 클러스터링** (KMeans, HAC, GMM, DBSCAN) = **24조합**
- 165개 type×family 주간 시계열, K=4 (SBC 4분류와 동일)
- FastDTW: `abs(a-b)` 거리 → MDS (Daiso 파이프라인), DBSCAN은 k-거리 중앙값 eps

### 품질 상위 5조합 (실루엣↑, DB↓)
| 순위 | 조합 | Silhouette | Davies-Bouldin | K |
|------|------|-----------|----------------|---|
| 1 | **TS2Vec+KMeans/HAC** | **0.921** | **0.235** | 4 |
| 2 | TS2Vec+GMM | 0.853 | 0.443 | 4 |
| 3 | PCA+HAC | 0.856 | 0.744 | 4 |
| 4 | FastDTW+HAC | 0.817 | 0.500 | 4 |
| 5 | AE+KMeans | 0.796 | 0.610 | 4 |

### 하위·주의
- **GAF-CNN**: 전 조합 실루엣 < 0.47 — 이미지 변환+얕은 CNN으로는 165 시계열 분리 부족
- **PatchTST**: 실루엣 0.43~0.53 — 임베딩 전용 학습(80 epoch) 한계
- **PCA+DBSCAN**: 유효 군집 1개 (eps 부적합)
- DBSCAN은 K≠4인 경우 다수 — 순위는 내부지표만, K=4 고정 실험과 직접 비교 불가

### 저장
- **최적 조합: TS2Vec+HAC** → `ml_cluster_type_family.parquet`
- 전체 24행 → `clustering_quality.csv`